## Handle OverBooking

Este proyecto tiene la finalidad de ayudar a Hoteles y empresas similares que gestionan sus reservas en Booking a evitar quedarse con habitaciones vacías, a través de estimar el overbooking en base a la probabilidad de cancelación de las reservas ya realizadas.

In [54]:
# Base de datos exploratoria, obtenida de Stack Overflow y guardad en la carpeta data en txt
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar la base de datos
df = pd.read_csv('../data/hotel_bookings.csv')

#Verificamos los valores nulos en el dataframe en 2 partes, primero de 0 a 20 y luego de 20 a 40
df.columns[0:20]
df.count()[0:20]


hotel                             119390
is_canceled                       119390
lead_time                         119390
arrival_date_year                 119390
arrival_date_month                119390
arrival_date_week_number          119390
arrival_date_day_of_month         119390
stays_in_weekend_nights           119390
stays_in_week_nights              119390
adults                            119390
children                          119386
babies                            119390
meal                              119390
country                           118902
market_segment                    119390
distribution_channel              119390
is_repeated_guest                 119390
previous_cancellations            119390
previous_bookings_not_canceled    119390
reserved_room_type                119390
dtype: int64

In [14]:
#Resto de columnas
df.columns[20:40]
df.count()[20:40]

assigned_room_type             119390
booking_changes                119390
deposit_type                   119390
agent                          103050
company                          6797
days_in_waiting_list           119390
customer_type                  119390
adr                            119390
required_car_parking_spaces    119390
total_of_special_requests      119390
reservation_status             119390
reservation_status_date        119390
dtype: int64

# Determinación de las variables

En base a lo observado, vamos a quitar las siguientes variables:

- agent: tiene nulos y no se considera reelevante para estimar la probabilidad de cancelación.
- reservation_status y reservation_status_date: Reflejan la misma información de la variable objetivo is_canceled y/o se gestionan al momento de ocurrir el evento.
- arrival_date_year: condiciona el modelo a tiempos pasados y evita generalizar.
- assigned_room_type: suele establecerse en el momento del checking físico por lo que no sería viable para predecir.

Variables a modificar:

- company: esta variable tiene sólo 6797 valores no nulos, pero de los 6797 que son company 5606 (82,48%) no se cancelaron, por lo que vamos a cambiar a la has_company (booleana).

In [ ]:
# Demostración del porqué debemos modificar la variable company a has_company y que sea boleana.

df_company = df[df['company'].notnull()]

print("Cantidad de reservas canceladas (Solo Empresas):")
print(df_company['is_canceled'].value_counts())



Cantidad de reservas canceladas (Solo Empresas):
is_canceled
0    5606
1    1191
Name: count, dtype: int64


# Preparación del DataFrame final

In [55]:
# Primero agregamos al dataframe original la variable has_company como una variable boleana en base a company.
df['has_company'] = df['company'].notnull()

# Eliminamos la columna company del dataframe original y las columnas que no nos interesan para el análisis.
df = df.drop(['company', 'agent', 'reservation_status', 'reservation_status_date', 'arrival_date_year', 'assigned_room_type'], axis=1)

# Eliminamos todos los nulos del dataframe original
df = df.dropna()

df.info()

<class 'pandas.DataFrame'>
Index: 118898 entries, 0 to 119389
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           118898 non-null  str    
 1   is_canceled                     118898 non-null  int64  
 2   lead_time                       118898 non-null  int64  
 3   arrival_date_month              118898 non-null  str    
 4   arrival_date_week_number        118898 non-null  int64  
 5   arrival_date_day_of_month       118898 non-null  int64  
 6   stays_in_weekend_nights         118898 non-null  int64  
 7   stays_in_week_nights            118898 non-null  int64  
 8   adults                          118898 non-null  int64  
 9   children                        118898 non-null  float64
 10  babies                          118898 non-null  int64  
 11  meal                            118898 non-null  str    
 12  country                         

## Split para entrenamiento